# 04 Model Experimentation

Comparacion curada de modelos sobre los splits temporales ya materializados en Snowflake. Este notebook queda dividido **por modelo** para que una caida de WSL no obligue a repetir toda la experimentacion.

## Enfoque de esta version

- el `model_zoo` queda reservado para experimentacion
- la produccion ya no se decide dentro de `src/models/train_model.py`; ahi se entrena solo el modelo seleccionado
- cada modelo corre en su propia celda
- despues de cada corrida se guarda progreso en `data/models/notebook04_progress.csv`

## Shortlist curado

Se removieron del flujo principal `ridge`, `bagging`, `pasting` y `voting` porque ya quedaron dominados en `notebooks/temp.txt`. `adaboost` tambien sale del shortlist principal porque no mejora frente a los boosters mas fuertes y solo agrega tiempo de ejecucion.

El shortlist que si vale la pena seguir comparando ahora es:

- `dummy_regressor`: baseline minimo
- `sgd_regressor`: evidencia real de entrenamiento incremental por lotes
- `random_forest`: referencia de ensamble no boosting
- `gradient_boosting`: mejor `val_rmse` completado en el log disponible
- `hist_gradient_boosting`: booster hist estable y relativamente eficiente
- `xgboost`, `lightgbm`, `catboost`: boosters modernos que siguen siendo candidatos reales si la dependencia y el entorno aguantan


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
import pandas as pd
from IPython.display import display

from src.data.ingestion import fetch_sample
from src.models.experiment_runner import prepare_experiment_context, run_single_experiment
from src.models.model_zoo import (
    CURATED_EXPERIMENT_MODEL_NAMES,
    available_model_entries,
    recommended_experiment_entries,
    unavailable_required_models,
)
from src.models.production_model import PRODUCTION_MODEL_NAME, PRODUCTION_SELECTION_EVIDENCE
from src.models.training_common import split_ranges
from src.utils.config import get_settings

pd.set_option('display.max_columns', 100)
settings = get_settings()
ranges = split_ranges(settings)
missing_required = unavailable_required_models(CURATED_EXPERIMENT_MODEL_NAMES)
available_names = [entry.name for entry in recommended_experiment_entries()]


In [ ]:
train_preview = fetch_sample(f"SELECT * FROM {settings.train_table}", limit=2000, settings=settings)
val_preview = fetch_sample(f"SELECT * FROM {settings.val_table}", limit=2000, settings=settings)
test_preview = fetch_sample(f"SELECT * FROM {settings.test_table}", limit=2000, settings=settings)
assert not train_preview.empty and not val_preview.empty and not test_preview.empty, 'Uno de los splits esta vacio.'

pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows_preview': [len(train_preview), len(val_preview), len(test_preview)],
    'min_pickup': [train_preview['pickup_datetime'].min(), val_preview['pickup_datetime'].min(), test_preview['pickup_datetime'].min()],
    'max_pickup': [train_preview['pickup_datetime'].max(), val_preview['pickup_datetime'].max(), test_preview['pickup_datetime'].max()],
})


In [ ]:
NOTEBOOK_TRAIN_SAMPLE_LIMIT = min(settings.train_sample_limit, 20000)
NOTEBOOK_SAMPLE_PCT = min(settings.train_sample_pct, 1.0)
SPARSE_EVAL_BATCH_SIZE = 25000
DENSE_EVAL_BATCH_SIZE = 5000
PROGRESS_PATH = PROJECT_ROOT / 'data' / 'models' / 'notebook04_progress.csv'
PROGRESS_PATH.parent.mkdir(parents=True, exist_ok=True)

experiment_params = pd.DataFrame({
    'parameter': [
        'NOTEBOOK_TRAIN_SAMPLE_LIMIT',
        'NOTEBOOK_SAMPLE_PCT',
        'SPARSE_EVAL_BATCH_SIZE',
        'DENSE_EVAL_BATCH_SIZE',
        'TRAINING_BATCH_GRAIN',
        'TRIP_TYPES',
        'CURATED_EXPERIMENT_MODEL_NAMES',
        'PRODUCTION_MODEL_NAME',
        'MISSING_REQUIRED_FROM_SHORTLIST',
    ],
    'value': [
        NOTEBOOK_TRAIN_SAMPLE_LIMIT,
        NOTEBOOK_SAMPLE_PCT,
        SPARSE_EVAL_BATCH_SIZE,
        DENSE_EVAL_BATCH_SIZE,
        settings.training_batch_grain,
        ','.join(settings.trip_types),
        ','.join(CURATED_EXPERIMENT_MODEL_NAMES),
        PRODUCTION_MODEL_NAME,
        ','.join(missing_required) if missing_required else 'none',
    ]
})
experiment_params


In [ ]:
context = prepare_experiment_context(
    settings=settings,
    sample_limit=NOTEBOOK_TRAIN_SAMPLE_LIMIT,
    sample_pct=NOTEBOOK_SAMPLE_PCT,
)

saved_results_df = pd.read_csv(PROGRESS_PATH) if PROGRESS_PATH.exists() else pd.DataFrame()
results = saved_results_df.to_dict(orient='records') if not saved_results_df.empty else []
trained_models = {}

print('Modelos disponibles en este entorno:', available_names)
print('Progreso persistido en:', PROGRESS_PATH)
if not saved_results_df.empty:
    display(saved_results_df.sort_values('val_rmse').reset_index(drop=True))


In [ ]:
def persist_results() -> pd.DataFrame:
    comparison = pd.DataFrame(results).sort_values('val_rmse').reset_index(drop=True)
    comparison.to_csv(PROGRESS_PATH, index=False)
    return comparison


def run_and_store(model_name: str, force: bool = False):
    global results
    existing = pd.DataFrame(results)
    if not existing.empty and model_name in existing['model'].tolist() and not force:
        print(f'SKIP {model_name}: ya existe en {PROGRESS_PATH.name}. Usa force=True si quieres recalcularlo.')
        return existing[existing['model'] == model_name].reset_index(drop=True)

    entries = available_model_entries([model_name])
    if not entries:
        print(f'SKIP {model_name}: dependencia no disponible en este entorno.')
        return None

    model, metrics = run_single_experiment(
        entries[0],
        context,
        sparse_eval_batch_size=SPARSE_EVAL_BATCH_SIZE,
        dense_eval_batch_size=DENSE_EVAL_BATCH_SIZE,
    )
    trained_models[model_name] = model
    results = [row for row in results if row.get('model') != model_name]
    results.append(metrics)
    comparison = persist_results()
    display(comparison[comparison['model'] == model_name].reset_index(drop=True))
    return comparison


## Baselines

Corre estas celdas una por una. Si una corrida termina bien, su metrica queda guardada en `notebook04_progress.csv`.


In [ ]:
run_and_store('dummy_regressor')


In [ ]:
run_and_store('sgd_regressor')


## Tree And Boosting Candidates

A partir de aqui quedan solo los candidatos que todavia justifican costo de ejecucion frente a lo observado en `temp.txt`.


In [ ]:
run_and_store('random_forest')


In [ ]:
run_and_store('gradient_boosting')


In [ ]:
run_and_store('hist_gradient_boosting')


In [ ]:
run_and_store('xgboost')


In [ ]:
run_and_store('lightgbm')


In [ ]:
run_and_store('catboost')


In [ ]:
comparison = pd.DataFrame(results).sort_values('val_rmse').reset_index(drop=True)
comparison


In [ ]:
best_row = comparison.iloc[0]
print('Selected model by validation:', best_row['model'])
print('Validation RMSE:', round(float(best_row['val_rmse']), 4))
print('Test RMSE:', round(float(best_row['test_rmse']), 4))
print('Current production model target:', PRODUCTION_MODEL_NAME)
print('Selection evidence for production script:', PRODUCTION_SELECTION_EVIDENCE)
print('Missing required shortlist dependencies:', missing_required)


## Conclusiones Operativas

- con la evidencia disponible al 2026-05-05, `gradient_boosting` ya tiene base suficiente para pasar a produccion: fue el mejor `val_rmse` completado del log y no depende de librerias externas inestables
- `catboost` sigue siendo un candidato experimental, pero no bloquea la ruta productiva actual porque la corrida quedo inconclusa por estabilidad de entorno, no por falta de implementacion
- el script productivo `python3 -m src.models.train_model` ya no compara el zoo: reentrena solo `gradient_boosting` y genera el artefacto estable `data/models/nyc_taxi_fare_production.joblib`
- si mas adelante quieres reabrir la decision, solo debes rerunear las celdas del shortlist y revisar `notebook04_progress.csv`
